# Sprint 2 — End-to-End Family Analysis

This notebook runs the existing GitSkills library from the prepared DuckDB database through family comparison, security scans, and review-sample generation. It uses candidate **families**, not manually selected artifact pairs. All source content is treated as data; nothing from an artifact is executed.

Run the cells top to bottom. Before validation begins, agree on and freeze the family list below.

## 1. Environment and paths

Use the project Python environment (see the [pipeline README](README.md)). `GITSKILLS_DB` points to the **prepared DuckDB** with `artifact_groupings` and `artifact_siblings`. Database connections made by the library are read-only. Results go into this notebook's `results/sprint2/` or `results/final/` folder; family-analysis reports remain under `apps/skilltrace/reports/`.

In [ ]:
import os
from pathlib import Path

import duckdb
import pandas as pd

from gitskills.analysis.family_models import SimilarityPolicy
from gitskills.pipeline import run_family, summarize_cohort

DEFAULT_DB_PATH = Path(r"C:\data\duckdb\agent_skills_release.db")
DB_PATH = Path(os.environ.get("GITSKILLS_DB", DEFAULT_DB_PATH)).expanduser()

PROJECT_ROOT = next(
    (directory for directory in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (directory / "pyproject.toml").is_file() and (directory / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from the project repository (or a directory beneath it).")
if not DB_PATH.is_file():
    raise FileNotFoundError(f"Prepared DuckDB not found: {DB_PATH}. Set GITSKILLS_DB.")

NOTEBOOK_DIR = PROJECT_ROOT / "notebooks" / "end_to_end_pipeline"
SKILLTRACE_REPORTS = PROJECT_ROOT / "apps" / "skilltrace" / "reports"
RUN_LABEL = "sprint2"  # Change to "final" for the final research run.
if RUN_LABEL not in {"sprint2", "final"}:
    raise ValueError("RUN_LABEL must be sprint2 or final")
RESULT_DIR = NOTEBOOK_DIR / "results" / RUN_LABEL

with duckdb.connect(str(DB_PATH), read_only=True) as connection:
    db_version = connection.execute("SELECT version()").fetchone()[0]
    available_tables = {row[0] for row in connection.execute("SHOW TABLES").fetchall()}
    required_tables = {"artifacts", "artifact_groupings", "artifact_siblings", "repos"}
    if missing := required_tables - available_tables:
        raise RuntimeError(f"Prepare the analysis database first; missing tables: {sorted(missing)}")
print(f"DuckDB {db_version} | {DB_PATH}")
print(f"Results: {RESULT_DIR.relative_to(PROJECT_ROOT)}")
print(f"Family reports: {SKILLTRACE_REPORTS.relative_to(PROJECT_ROOT)}")

## 2. Select candidate families

Edit the list as the study evolves. Each name is matched exactly against `artifact_groupings.name`. The existing SkillTrace reports are a starting cohort, **not an automatically approved research sample**. Set `APPROVAL_REFERENCE` to the team decision/date after approval; the manifest otherwise marks the cohort provisional. All directed edges (primary and additional) are included; ambiguous relationships remain eligible only for relationship validation.

In [ ]:
candidate_families = [
    "003-create-golang-project",
    "1k-defi-module-integration",
    "A_B_Test_Setup-934740a5939a6121",
    "argent-react-native-app-workflow",
    "busybox-on-windows",
    "capacitor-best-practices",
    "cometchat-core",
    "databricks-upgrade-migration",
    "legacy-migration-planner",
]

POLICY = SimilarityPolicy()  # Shared similarity/direction thresholds; see RDR-004 and RDR-007.
EDGES = "all"  # Both primary and additional direction-supported comparisons.
PER_STRATUM = 5
SEED = 580
APPROVAL_REFERENCE = None  # Set to a recorded team decision/date after approval.

if len(candidate_families) != len(set(candidate_families)) or not candidate_families:
    raise ValueError("Use a nonempty list of unique family names")
print(f"Selected families: {len(candidate_families)}")

## 3. Analyze and scan each family

`run_family()` loads the family and siblings from DuckDB, refreshes a missing/incompatible `analyze_family` report in SkillTrace, then writes a current `scan_family` report to `results/<run>/scans/`. It uses the shared library; no CLI calls or notebook-specific detection logic. Skipped artifacts and incomplete coverage are retained with reasons. You can run this cell for one family at a time by temporarily narrowing `candidate_families`.

In [ ]:
runs = []
for family_name in candidate_families:
    record = run_family(
        family_name, database=DB_PATH, skilltrace_reports=SKILLTRACE_REPORTS,
        scans=RESULT_DIR / "scans", policy=POLICY, edges=EDGES,
    )
    runs.append(record)
    print(f"{family_name}: {record['artifacts_loaded']} loaded, "
          f"{record['artifacts_skipped']} skipped, {record['pairs_analyzed']} pairs")

display(pd.DataFrame(runs)[[
    "family", "artifacts_loaded", "artifacts_skipped", "pairs_analyzed",
    "pairs_incomplete", "family_report_updated",
]])

## 4. Aggregate the selected cohort

The summary uses **only** the names above, even if earlier reports remain in `scans/`. It verifies source identifiers, matches scan inputs to the SkillTrace reports, rejects mismatched settings, and writes the cohort manifest, audit files, research tables, figure, and two deterministic manual-review samples. Reports for any unprocessed family must be generated with step 3 first.

In [ ]:
execution = summarize_cohort(
    candidate_families, database=DB_PATH, skilltrace_reports=SKILLTRACE_REPORTS,
    results=RESULT_DIR, per_stratum=PER_STRATUM, seed=SEED,
    approval_reference=APPROVAL_REFERENCE,
)
display(pd.DataFrame([execution]))
print(f"Research summary: {RESULT_DIR / 'review' / 'summary.json'}")
print(f"Sample manifest: {RESULT_DIR / 'sample_manifest.json'}")
print(f"Processing audit: {RESULT_DIR / 'audit' / 'processing_summary.csv'}")

## 5. Independent review and adjudication (team action)

The pipeline is complete when the outputs above are written; **human validation is separate**. Follow [`docs/validation/manual_validation_protocol.md`](../../docs/validation/manual_validation_protocol.md):

1. Give each reviewer an independent copy of `review/validation_annotations_template.csv` and `review/relationship_validation_annotations_template.csv`.
2. Inspect source artifacts and siblings using the IDs and paths in `sample_manifest.json` and SkillTrace. Do not run stored scripts.
3. Preserve individual reviews and resolve disagreements in separate adjudicated CSVs (recommended: `review/adjudicated/`). Do not overwrite generated templates.
4. Once both reviews are adjudicated, run the optional cell below to add sample-level validation metrics to the research summary.

If the cohort, policy, or rules change after review begins, archive the old results and validation copies before creating a new sample.

In [ ]:
directed_review = RESULT_DIR / "review" / "adjudicated" / "directed.csv"
relationship_review = RESULT_DIR / "review" / "adjudicated" / "relationships.csv"

if directed_review.is_file() and relationship_review.is_file():
    completed = summarize_cohort(
        candidate_families, database=DB_PATH, skilltrace_reports=SKILLTRACE_REPORTS,
        results=RESULT_DIR, per_stratum=PER_STRATUM, seed=SEED,
        approval_reference=APPROVAL_REFERENCE,
        directed_annotations=directed_review,
        relationship_annotations=relationship_review,
    )
    print("Adjudicated sample metrics written to review/summary.json")
else:
    print("Manual review is pending. See the validation protocol and copy the templates for reviewers.")